# Load Models and Embeddings

In [ ]:
# Load environment variables and models
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path="C:/Support-Ticket-Classifier-with-RAG/key.env")
#load_dotenv()  
groq_token = os.getenv("GROQ_API_KEY")

# Loading the LLM model (Llama 3.1)
from langchain_groq import ChatGroq
groq_model = ChatGroq(api_key=groq_token, model="llama-3.1-8b-instant")

# Loading the HuggingFace embeddings model
from langchain_huggingface import HuggingFaceEmbeddings
embeds = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Support-Ticket-Classifier-with-RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2223.79it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Define Knowledge Base and Save to File

In [3]:
import os

# Function to save the knowledge base to a file
def save_knowledge_base(data, file_path):
    # Extract directory from the file path
    directory = os.path.dirname(file_path)

    # Check if the directory exists, if not, create it
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory '{directory}' created.")
    else:
        print(f"Directory '{directory}' already exists.")

    # Write the knowledge base data to the file
    with open(file_path, "w") as file:
        for entry in data:
            file.write(entry + "\n")
        print(f"Knowledge base saved to {file_path}")

# Define the knowledge base content
knowledge_base = [
    "Category 1 - Login Issues - Login issues often occur due to incorrect passwords or account lockouts.",
    "Category 2 - App Functionality - App crashes can be caused by outdated software or device incompatibility.",
    "Category 3 - Billing - Billing discrepancies may result from processing errors or duplicate transactions.",
    "Category 4 - Account Management - Account management includes tasks such as changing profile information, linking social media accounts, and managing privacy settings.",
    "Category 5 - Performance Issues - Performance issues can be related to device specifications, network connectivity, or app optimization."
]

# Path to store the knowledge base
kb_file_path = "C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt"

# Save the knowledge base to the file
save_knowledge_base(knowledge_base, kb_file_path)


Directory 'C:/Support-Ticket-Classifier-with-RAG/data' already exists.
Knowledge base saved to C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt


# Load and Split the Knowledge Base into Chunks

In [4]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Function to load and split documents
def load_and_split_docs(file_path, chunk_size=200):
    loader = TextLoader(file_path)
    docs = loader.load()  # Loading documents from the file
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size)
    return splitter.split_documents(docs)

# Load and split the knowledge base into chunks
kb_chunks = load_and_split_docs(kb_file_path)

In [5]:
kb_chunks

[Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt'}, page_content='Category 1 - Login Issues - Login issues often occur due to incorrect passwords or account lockouts.'),
 Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt'}, page_content='Category 2 - App Functionality - App crashes can be caused by outdated software or device incompatibility.'),
 Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt'}, page_content='Category 3 - Billing - Billing discrepancies may result from processing errors or duplicate transactions.'),
 Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-RAG/data/knowledge_base.txt'}, page_content='Category 4 - Account Management - Account management includes tasks such as changing profile information, linking social media accounts, and managing privacy settings.'),
 Document(metadata={'source': 'C:/Support-Ticket-Classifier-with-R

# Create a Vector Store

In [6]:
from langchain_community.vectorstores import FAISS

# Function to create a vector store
def create_vector_store(documents, embeddings_model):
    return FAISS.from_documents(documents=documents, embedding=embeddings_model)

# Create the vector store using the knowledge base chunks and embeddings model
kb_vectorstore = create_vector_store(kb_chunks, embeds)

# Define the System Prompt for Question-Answering

In [47]:
# Importing the ChatPromptTemplate for prompt creation
from langchain_core.prompts import ChatPromptTemplate

# System prompt providing instructions and context
guidelines_prompt = (
"""
### Guidelines

Background Information:
{context}

You are a support ticket assistant.

Follow these steps:

1. Analyze the given input text carefully.
2. Classify it into ONE of these categories:
   - Category 1 - Login Issues
   - Category 2 - App Functionality
   - Category 3 - Billing
   - Category 4 - Account Management
   - Category 5 - Performance Issues

3. Provide a short and simple solution (max 2-3 lines) for the issue.

4. If the input is unrelated or unclear, respond with:
   Category: I don't know
   Solution: Cannot determine a proper solution

### Output Format (STRICT)

You must return ONLY the following format:

Category: Category X - <Category Name>
Solution: <short solution (max 2-3 lines)>

Rules:
- No extra text before or after
- No explanation
- No bullet points
- No repetition of the question
- If unsure: Category: I don't know
  Solution: Cannot determine a solution
  
"""
)

qa_template = ChatPromptTemplate.from_messages(
    [
        ("system", guidelines_prompt),
        ("human", "{input}"),
    ]
)


# Build the Retrieval-Augmented Generation Chain

In [48]:
# Importing required function to create QA chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains import create_retrieval_chain

# Create the QA chain using the vector store and prompt template
qa_chain = create_stuff_documents_chain(groq_model, qa_template)

# Create the retriever for the vector store
retriever = kb_vectorstore.as_retriever(k=3)

# Create the retrieval chain combining vector store and QA chain
rag_chain = create_retrieval_chain(retriever, qa_chain)

# Classify Support Tickets

In [50]:
# Function to classify support tickets using the RAG model
def classify_support_tickets(tickets, rag_chain):
    classified_tickets = []

    # Process each ticket and store the classification result
    for ticket in tickets:
        # Extract the ticket text
        ticket_text = ticket['text']

        # Invoke the RAG chain to classify the ticket
        response = rag_chain.invoke({"input": ticket_text})

        # Append the classification result
        classified_tickets.append({"ticket": ticket_text,"classification": response['answer']})

    return classified_tickets

def display_classified_tickets(classified_tickets):
    print("\n=== 📌 Classified Support Tickets Report ===\n")

    for i, ticket_info in enumerate(classified_tickets, 1):
        print(f"🎫 Ticket {i}")
        print("-" * 50)
        print(f"📝 Issue:", ticket_info['ticket'])

        result = ticket_info['classification']
        category = "N/A"
        solution = "N/A"

        if "Category:" in result:
            try:
                category = result.split("Category:")[1].split("Solution:")[0].strip()
            except:
                pass

        if "Solution:" in result:
            try:
                solution = result.split("Solution:")[1].strip()
            except:
                pass

        # Clean output
        print(f"🏷 Category:", category)
        print(f"💡 Solution:", solution)
        print("\n" + "=" * 60 + "\n")

# Example tickets
support_tickets = [
    {"text": "My account login is not working. I've tried resetting my password twice."},
    {"text": "The app crashes every time I try to upload a photo."},
    {"text": "I was charged twice for my last subscription payment."},
    {"text": "I can't find the option to change my profile picture."},
    {"text": "The video playback is very laggy on my device."}
]

# Classify the support tickets using the RAG chain
classified_tickets = classify_support_tickets(support_tickets, rag_chain)

# Display the classified tickets
display_classified_tickets(classified_tickets)


=== 📌 Classified Support Tickets Report ===

🎫 Ticket 1
--------------------------------------------------
📝 Issue: My account login is not working. I've tried resetting my password twice.
🏷 Category: Category 1 - Login Issues
💡 Solution: Check your email for password reset links or contact our support team for further assistance.


🎫 Ticket 2
--------------------------------------------------
📝 Issue: The app crashes every time I try to upload a photo.
🏷 Category: Category 2 - App Functionality
💡 Solution: Try updating your app to the latest version or check your device compatibility.


🎫 Ticket 3
--------------------------------------------------
📝 Issue: I was charged twice for my last subscription payment.
🏷 Category: Category 3 - Billing
💡 Solution: Contact our billing department to investigate and cancel the duplicate charge. A refund will be processed if necessary.


🎫 Ticket 4
--------------------------------------------------
📝 Issue: I can't find the option to change my prof